# GIAI ĐOẠN 6: CHẨN ĐOÁN PHẦN DƯ, DIỄN GIẢI MÔ HÌNH ĐA PHƯƠNG PHÁP (XAI) & MÔ PHỎNG CHÍNH SÁCH SURGE CAP
**Đề tài:** Phân tích Biểu hiện Định giá Động (*Dynamic Surge Pricing*) và Dự báo Giá báo API trên Nền tảng Gọi xe Công nghệ (Uber & Lyft Boston, MA)  
**Mã đề tài:** `PRJ-EXT-01` | **Nhóm thực hiện:** Nhóm 02 — Trần Thanh Huy (`2521001060`) & Lê Huy Thắng (`2521001092`)  
**Môi trường thực thi:** Python `3.11.9` | `pandas 3.1.1` | `numpy 2.4.6` | `scipy 1.17.1` | `statsmodels 0.15.0` | `scikit-learn 1.9.1`

---
## 1. Chẩn đoán Phần dư Nâng cao (M7 HistGradientBoosting vs M2 OLS Tương tác — Tập Test `n = 46,573`)
- **M7 (`HistGradientBoosting`):** Mean = `+$0.0117`, SD = `$1.7830`, Skewness = `3.1060`, Kurtosis = `27.8009` (giảm quá nửa sau khi bỏ nhiễu `fillna(1.0)` trên Lyft), Breusch-Pagan LM = `610.62` ($p < 0.0001$). Tỷ lệ dự báo lệch trong $\pm \$1.00$: `60.42%`, trong $\pm \$2.00$: **`85.87%`**, trong $\pm \$3.00$: **`94.14%`**.
- **M2 (`Interaction OLS`):** Mean = `+$0.0120`, SD = `$1.9802`, tỷ lệ trong $\pm \$2.00$: `82.91%`, trong $\pm \$3.00$: `92.83%`.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))

from config import BASE_CAT_FEATURES, MODELS_DIR, TARGET_COL  # noqa: E402
from evaluation import calculate_metrics  # noqa: E402
from features import (  # noqa: E402
    create_data_splits,
    engineer_features_stage4,
    get_feature_columns_for_modeling,
    load_master_dataset,
)

# Kiểm chứng trực tiếp best_model.joblib (M7) trên tập Test 46,573 dòng
df_m_live = engineer_features_stage4(load_master_dataset())
_, te_idx_live = create_data_splits(df_m_live)[0]["random_80_20"]
feat_cols = get_feature_columns_for_modeling()
X_te_live = df_m_live[feat_cols].iloc[te_idx_live].copy()
for c in BASE_CAT_FEATURES:
    X_te_live[c] = X_te_live[c].astype("category")
y_te_live = df_m_live[TARGET_COL].iloc[te_idx_live].astype(np.float64).values
m7_live = joblib.load(MODELS_DIR / "best_model.joblib")
m7_metrics = calculate_metrics(y_te_live, m7_live.predict(X_te_live))
print(f"[Live Verify M7 on Test {len(y_te_live):,}]:", m7_metrics, "\n")

seg_df = pd.read_csv(BASE_DIR / "reports/tables/stage6_segment_errors.csv")
print("=== SAI SỐ THEO 12 PHÂN KHÚC DỊCH VỤ (SERVICE TIER) ===")
seg_cols = [
    "segment_value",
    "n_test",
    "mean_actual_usd",
    "m7_mae_usd",
    "m7_rmse_usd",
    "m7_mape_pct",
    "m7_within_2usd_pct",
]
tier_seg = seg_df[seg_df["dimension"] == "service_tier"][seg_cols]
print(tier_seg.to_string(index=False))

2026-11-28 03:42:30,631 - INFO - Đã nạp master_rideshare.parquet: 270,898 dòng x 32 cột.


2026-11-28 03:42:30,762 - INFO - Đang thiết lập 3 chiến lược chia dữ liệu (test_size=0.2, random_state=42)...


[Live Verify M7 on Test 46,573]: {'R2': 0.9626, 'Adjusted_R2': 0.9626, 'RMSE': 1.783, 'MAE': 1.1338, 'MAPE_pct': 8.19, 'Max_Error': 29.9885} 

=== SAI SỐ THEO 12 PHÂN KHÚC DỊCH VỤ (SERVICE TIER) ===
segment_value  n_test  mean_actual_usd  m7_mae_usd  m7_rmse_usd  m7_mape_pct  m7_within_2usd_pct
        Black    4712           20.679      1.2460       1.8153         5.79               81.39
    Black SUV    4705           30.310      1.3834       2.1623         4.41               77.41
          Lux    3021           17.834      1.0116       1.3828         5.59               84.77
    Lux Black    3028           22.882      1.0927       1.5305         4.60               82.89
 Lux Black XL    3125           32.331      1.4175       1.8699         4.27               75.90
         Lyft    3091            9.629      0.6004       0.8702         6.23               96.18
      Lyft XL    3146           15.321      1.0082       1.2711         6.76               90.27
       Shared    3029    

## 2. Khoảng Dự báo Bất định Quantile `[P10, P50, P90]` (dùng `pandas category` gốc) & Monotonic Split-CQR
- **Bản GĐ6 (100% Train, `pandas category` gốc):** Tỷ lệ bao phủ chặt `PICP = 78.44%` (dung sai $10^{-6}$ USD: `80.28%`), độ rộng trung bình `MPIW = $3.51`.
- **Hiệu chỉnh Monotonic Split-CQR (`85% fit / 15% cal`):** Đạt **`79.89%`** toàn tập (cả 12/12 gói hội tụ trong `[78.62%, 81.99%]`, `MPIW = $3.562`, **`0`** cắt phân vị ngược).

In [2]:
pi_df = pd.read_csv(BASE_DIR / "reports/tables/stage6_prediction_interval.csv")
print(pi_df.to_string(index=False))

cab_type service_tier  n_test  mean_actual_usd  mean_p10_usd  mean_p50_usd  mean_p90_usd  mpiw_width_usd  relative_width_pct  picp_coverage_pct
    Lyft       Shared    3029             6.03          4.11          6.35          7.86           3.742               62.09              80.79
    Lyft         Lyft    3091             9.63          8.67          9.56         10.62           1.946               20.21              73.76
    Lyft      Lyft XL    3146            15.32         13.76         15.16         17.03           3.273               21.36              79.59
    Lyft          Lux    3021            17.83         15.95         17.73         19.57           3.622               20.31              82.03
    Lyft    Lux Black    3028            22.88         20.93         22.62         24.52           3.592               15.70              81.90
    Lyft Lux Black XL    3125            32.33         29.13         32.06         34.36           5.228               16.17            

## 3. Diễn giải Mô hình Đa phương pháp: Biểu giá 12 Gói xe & Permutation Importance


In [3]:
tar_df = pd.read_csv(BASE_DIR / "reports/tables/stage6_tier_tariffs.csv")
tar_cols = [
    "cab_type",
    "service_tier",
    "base_fare_b0_usd",
    "per_mile_rate_b1_usd",
    "expected_fare_1mi_usd",
    "expected_fare_2_5mi_usd",
    "expected_fare_5mi_usd",
]
print(tar_df[tar_cols].to_string(index=False))

cab_type service_tier  base_fare_b0_usd  per_mile_rate_b1_usd  expected_fare_1mi_usd  expected_fare_2_5mi_usd  expected_fare_5mi_usd
    Lyft       Shared            3.3439                1.2377                   4.58                     6.44                   9.53
    Lyft         Lyft            5.4039                1.8387                   7.24                    10.00                  14.60
    Lyft      Lyft XL            7.4918                3.4114                  10.90                    16.02                  24.55
    Lyft          Lux            8.5244                4.0487                  12.57                    18.65                  28.77
    Lyft    Lux Black           11.7449                4.9637                  16.71                    24.15                  36.56
    Lyft Lux Black XL           20.7326                5.0902                  25.82                    33.46                  46.18
    Uber     UberPool            6.1400                1.1862        

In [4]:
perm_df = pd.read_csv(
    BASE_DIR / "reports/tables/stage6_permutation_importance.csv"
)
perm_cols = [
    "feature_group",
    "domain_family",
    "m7_delta_rmse_mean_usd",
    "m7_importance_share_pct",
    "m2_delta_rmse_mean_usd",
]
print(perm_df[perm_cols].to_string(index=False))

                               feature_group                domain_family  m7_delta_rmse_mean_usd  m7_importance_share_pct  m2_delta_rmse_mean_usd
                    name (Phân khúc dịch vụ)         1. Dịch vụ & Hãng xe                 10.0207                    69.10                  9.8826
    distance × surge (Tương tác cự ly–surge) 3. Định giá Động & Tương tác                  3.2669                    22.53                  0.3357
       source & destination (Điểm đón & trả)  2. Khoảng cách & Không gian                  0.7311                     5.04                  0.0568
surge_multiplier & lyft_excess (Hệ số surge) 3. Định giá Động & Tương tác                  0.3154                     2.17                  0.2668
   distance & log_distance (Quãng đường gốc)  2. Khoảng cách & Không gian                  0.1493                     1.03                  3.2292
      cab_type & is_lyft (Hãng xe Uber/Lyft)         1. Dịch vụ & Hãng xe                  0.0139                     

## 4. Mô phỏng Chính sách Kinh tế Trần Giá Linh hoạt dưới Ràng buộc Cung (Surge Cap Policy Simulation - RQ4)
- **Nguyên tắc phân phối bên ngắn (*Short-Side Rationing*, $Q_{\text{realized}} = \min(Q_D, Q_S)$)** trên `92,384` lượt Lyft quan sát surge thực (tổng cơ sở `$1,603,696.20` API Quote Proxy), kết hợp đối chứng cận trên co giãn cầu ($\epsilon_d \in \{-0.50, -0.80\}$, Cohen et al., 2016) và M2 OLS.

In [5]:
sim_df = pd.read_csv(
    BASE_DIR / "reports/tables/stage6_surge_cap_simulation.csv"
)
cols = [
    "scenario_code",
    "surge_cap",
    "affected_trips_count",
    "mean_rider_saving_per_affected_trip_usd",
    "m2_ols_mean_saving_affected_usd",
    "static_lyft_rev_change_pct",
    "supply_constrained_rev_change_drop5_pct",
    "supply_constrained_rev_change_drop10_pct",
    "net_lyft_rev_change_eps_050_pct",
]
print(sim_df[cols].to_string(index=False))

        scenario_code  surge_cap  affected_trips_count  mean_rider_saving_per_affected_trip_usd  m2_ols_mean_saving_affected_usd  static_lyft_rev_change_pct  supply_constrained_rev_change_drop5_pct  supply_constrained_rev_change_drop10_pct  net_lyft_rev_change_eps_050_pct
Baseline_No_Cap_3.00x       3.00                     0                                     0.00                             0.00                       0.000                                    0.000                                     0.000                            0.000
       Cap_2.00x_Mild       2.00                    53                                     5.53                             5.88                      -0.018                                   -0.023                                    -0.027                           -0.011
   Cap_1.75x_Balanced       1.75                   704                                     4.61                             5.14                      -0.203                         